# Gemma 4 E4B Fine-Tuning for Hable Ya

Fine-tune Gemma 4 E4B on SFT and DPO datasets generated from eval fixtures.

**Prerequisites:**
- Generate datasets: `python -m finetune.generate`
- Download HF weights: `python scripts/download_model.py --hf-only`
- GPU with >= 16 GB VRAM (Unsloth 4-bit)

**Outputs:**
- LoRA adapter saved to `models/gemma-4-e4b-lora/`
- Merged GGUF exported to `models/gemma-4-e4b-finetuned.gguf`

In [ ]:
# Install dependencies (run once)
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install datasets trl

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

HF_TOKEN = os.environ["HF_TOKEN"]

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # assumes running from notebooks/
DATASETS_DIR = REPO_ROOT / "finetune" / "datasets"
MODELS_DIR = REPO_ROOT / "models"
OUTPUT_DIR = MODELS_DIR / "gemma-4-e4b-lora"

MODEL_NAME = "unsloth/gemma-4-E4B-it"
LOCAL_WEIGHTS = MODELS_DIR / "gemma-4-e4b-hf"

# Use local weights if already downloaded, otherwise pull from HF
MODEL_PATH = str(LOCAL_WEIGHTS) if LOCAL_WEIGHTS.exists() else MODEL_NAME
print(f"Model source: {MODEL_PATH}")

## 1. Load Model with Unsloth

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # Turn off for just text!
    finetune_language_layers=True,  # Should leave on!
    finetune_attention_modules=True,  # Attention good for GRPO
    finetune_mlp_modules=True,  # Should leave on always!
    r=16,  # Larger = higher accuracy, but might overfit
    lora_alpha=16,  # Recommended alpha == r at least
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
model.print_trainable_parameters()

## 2. Load SFT Dataset

In [ ]:
import json


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


sft_records = load_jsonl(DATASETS_DIR / "sft_train.jsonl")
print(f"Loaded {len(sft_records)} SFT examples")
print(f"Keys: {list(sft_records[0].keys())}")
print(f"Roles: {[m['role'] for m in sft_records[0]['messages']]}")

In [ ]:
from collections import Counter

bands = Counter(r["metadata"]["cefr_band"] for r in sft_records)
categories = Counter(r["metadata"]["category"] for r in sft_records)

print("CEFR distribution:")
for band in ["A1", "A2", "B1", "B2", "C1"]:
    print(f"  {band}: {bands.get(band, 0)}")

print("\nCategory distribution:")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

In [ ]:
from datasets import Dataset


def format_for_training(record: dict) -> dict:
    """Apply the chat template to convert messages into a single text field."""
    text = tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


sft_dataset = Dataset.from_list(sft_records).map(format_for_training)
print(f"Dataset size: {len(sft_dataset)}")
print(f"\nSample (first 500 chars):\n{sft_dataset[0]['text'][:500]}")

## 3. SFT Training

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,  # if memory allows; helps diversity per step
        gradient_accumulation_steps=4,  # effective batch = 8
        warmup_ratio=0.03,  # warmup_steps=5 is too abrupt at this LR
        num_train_epochs=2,
        learning_rate=2e-5,  # was 2e-4 — an order of magnitude down
        logging_steps=10,
        save_strategy="epoch",  # checkpoint each epoch so you can A/B
        optim="adamw_8bit",
        weight_decay=0.01,  # was 0.001 — light regularization
        lr_scheduler_type="cosine",  # gentler than linear at the end
        seed=3407,
        report_to="none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

In [ ]:
sft_stats = sft_trainer.train()
print(f"\nTraining loss: {sft_stats.training_loss:.4f}")

## 4. Quick Inference Check

Sanity-check the fine-tuned model before continuing to DPO.

In [ ]:
FastModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are a Spanish conversation partner for language learners.\nThe learner is at CEFR level A2 (production_level=0.35).\nWhen the learner makes an error, recast it naturally in your response -- never correct explicitly. Keep responses to 1-3 sentences with exactly one question. After responding, call log_turn with the learner utterance, errors, and fluency indicators.",
            }
        ],
    },
    {
        "role": "assistant",
        "content": [{"type": "text", "text": "Hola, \u00bfqu\u00e9 hiciste hoy?"}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Yo fui al tienda y compr\u00e9 mucho frutas."}
        ],
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1] :], skip_special_tokens=True)
print("Model response:\n")
print(response)

## 6. Save and Export

Save the final LoRA adapter and export a merged GGUF for llama.cpp deployment.

In [ ]:
# Save final adapter
final_adapter_path = OUTPUT_DIR / "final-adapter"
model.save_pretrained(str(final_adapter_path))
tokenizer.save_pretrained(str(final_adapter_path))
print(f"Final adapter saved to {final_adapter_path}")

In [ ]:
model.save_pretrained_gguf(
    MODELS_DIR / "gemma-4-e4b-finetuned",
    tokenizer,
    quantization_method="Q8_0",  # For now only Q8_0, BF16, F16 supported
)

In [ ]:
# Upload to HF
model.push_to_hub_merged("adrianmoses/gemma-4-e4b-hable-ya", tokenizer, token=HF_TOKEN)

In [ ]:
model.push_to_hub_gguf(
    "adrianmoses/gemma-4-e4b-hable-ya-gguf",
    tokenizer,
    quantization_method="Q8_0",
    token=HF_TOKEN,
)